# Partie III — RNN, LSTM, GRU et Seq2Seq
### Projet de fin de module — Deep Learning — EMSI Casablanca (2025–2026)

**Thème :** Modélisation de séquences et génération/traduction sur données textuelles réelles.

**Datasets réels utilisés :**
- **Partie A — Modèle de langage (next-token prediction) :** corpus **IMDb** (critiques de films en anglais), via la librairie `datasets` (HuggingFace).
- **Partie B — Seq2Seq (traduction automatique) :** corpus parallèle **Tatoeba / eng-fra** (corpus classique anglais → français), téléchargé directement dans Colab.

**Plan du notebook (aligné sur le cahier des charges, section 7) :**

1. Objectif probabiliste d'un modèle de langage — règle de chaîne
2. Perplexité — définition et interprétation
3. Préparation des données (Partie A — IMDb)
4. Implémentation successive : RNN simple → LSTM → GRU
5. BPTT et gradient clipping (illustration expérimentale)
6. Comparaison RNN / LSTM / GRU (stabilité, performance, mémoire, coût)
7. Préparation des données (Partie B — Tatoeba eng-fra)
8. Architecture encodeur–décodeur et *teacher forcing*
9. Stratégies de décodage : glouton et *beam search*
10. Évaluation (perplexité, BLEU)
11. Analyse critique
12. Question de synthèse — Partie III

> **Remarque pratique (Colab) :** ce notebook a besoin d'Internet (déjà actif sur Colab) pour télécharger les deux datasets. Activez un GPU : `Exécution > Modifier le type d'exécution > GPU`.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Installation et imports
# ---------------------------------------------------------------------------
!pip install -q datasets nltk

import os, re, math, time, random, zipfile, urllib.request
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import matplotlib.pyplot as plt
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)


## 1. Objectif probabiliste d'un modèle de langage

Un modèle de langage cherche à modéliser la distribution de probabilité d'une séquence de tokens
$x_1, x_2, \dots, x_T$. En appliquant la **règle de chaîne** (*chain rule*) des probabilités, on factorise
la probabilité jointe en un produit de probabilités conditionnelles :

$$P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1})$$

C'est précisément ce que modélise un réseau récurrent : à chaque pas de temps $t$, le réseau reçoit
le token précédent (et son état caché résumant le passé) et produit une distribution
$P(x_t \mid x_{<t}) = \text{softmax}(W h_t + b)$ sur le vocabulaire.

**Fonction de coût :** on entraîne le modèle par maximum de vraisemblance, ce qui revient à minimiser la
**négative log-vraisemblance (NLL)**, équivalente à l'entropie croisée moyenne par token :

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})$$

## 2. Perplexité — définition et interprétation

La **perplexité** (PPL) est l'exponentielle de la NLL moyenne par token :

$$\text{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})\right) = \exp(\mathcal{L})$$

**Interprétation :** la perplexité mesure le facteur de branchement effectif moyen du modèle, c'est-à-dire
le nombre moyen de choix équiprobables que le modèle "hésite" à faire à chaque position. Une perplexité de 1
correspond à une prédiction parfaite ; une perplexité égale à la taille du vocabulaire correspond à un modèle
aussi mauvais qu'un tirage uniforme aléatoire. Plus la perplexité est basse, meilleur est le modèle de langage.


## 3. Préparation des données — Partie A : modèle de langage (IMDb)

On utilise le corpus **IMDb** (critiques de films, dataset réel) en tâche de **prédiction du prochain token**
(*next-token prediction*) : on découpe chaque critique en une séquence de mots, et le modèle doit prédire,
à chaque position, le mot suivant.

Étapes : tokenisation, construction du vocabulaire (avec seuil de fréquence min), tokens spéciaux
`<pad>`, `<unk>`, `<bos>`, `<eos>`, troncature/découpage en fenêtres de longueur fixe, padding et masquage,
puis mini-lots (mini-batches) via `DataLoader`.


In [ ]:
# ---------------------------------------------------------------------------
# 3.1 Chargement du dataset réel IMDb
# ---------------------------------------------------------------------------
from datasets import load_dataset

# NB : l'ancien identifiant "imdb" (sans namespace) n'est plus resolu par les
# versions recentes de `datasets`/`huggingface_hub` (qui exigent un id
# "namespace/nom"). Le dataset est desormais heberge sous "stanfordnlp/imdb".
try:
    imdb = load_dataset("stanfordnlp/imdb")
except Exception:
    imdb = load_dataset("imdb")  # fallback si l'ancien identifiant fonctionne encore
# Pour limiter le temps de calcul (projet pédagogique), on sous-échantillonne
N_TRAIN, N_VAL, N_TEST = 4000, 800, 800
train_texts = imdb["train"].shuffle(seed=SEED)["text"][:N_TRAIN]
val_texts   = imdb["test"].shuffle(seed=SEED)["text"][:N_VAL]
test_texts  = imdb["test"].shuffle(seed=SEED+1)["text"][N_VAL:N_VAL+N_TEST]

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print(train_texts[0][:300])


In [ ]:
# ---------------------------------------------------------------------------
# 3.2 Tokenisation simple et vocabulaire avec tokens speciaux
# ---------------------------------------------------------------------------
TOKEN_RE = re.compile(r"[A-Za-z']+")

def tokenize(text):
    text = re.sub(r"<br\s*/?>", " ", text)
    return TOKEN_RE.findall(text.lower())

PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"
SPECIALS = [PAD, UNK, BOS, EOS]

class Vocab:
    def __init__(self, token_lists, min_freq=3, specials=SPECIALS):
        counter = Counter(tok for toks in token_lists for tok in toks)
        self.itos = list(specials) + [w for w, c in counter.items() if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, tokens):
        unk_id = self.stoi[UNK]
        return [self.stoi.get(t, unk_id) for t in tokens]

    def __len__(self):
        return len(self.itos)

train_tokens = [tokenize(t) for t in train_texts]
vocab = Vocab(train_tokens, min_freq=3)
print("Taille du vocabulaire :", len(vocab))


In [ ]:
# ---------------------------------------------------------------------------
# 3.3 Dataset de prediction du prochain token : decoupage en fenetres,
#     padding et masquage
# ---------------------------------------------------------------------------
MAX_LEN = 60  # longueur de sequence (fenetre) pour le LM

class LMDataset(Dataset):
    # Decoupe chaque texte en fenetres de longueur MAX_LEN.
    # input  = [<bos>, w1, ..., w_{L-1}]
    # target = [w1, ..., w_{L-1}, <eos>]   (decale d'un token : next-token prediction)
    def __init__(self, texts, vocab, max_len=MAX_LEN):
        self.vocab = vocab
        self.max_len = max_len
        self.sequences = []
        bos_id, eos_id = vocab.stoi[BOS], vocab.stoi[EOS]
        for text in texts:
            ids = vocab.encode(tokenize(text))[: max_len - 2]
            if len(ids) < 2:
                continue
            self.sequences.append([bos_id] + ids + [eos_id])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        x = torch.tensor(seq[:-1], dtype=torch.long)
        y = torch.tensor(seq[1:],  dtype=torch.long)
        return x, y

def lm_collate(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    x_pad = pad_sequence(xs, batch_first=True, padding_value=vocab.stoi[PAD])
    y_pad = pad_sequence(ys, batch_first=True, padding_value=vocab.stoi[PAD])
    mask = (x_pad != vocab.stoi[PAD]).float()  # masque de padding
    return x_pad, y_pad, mask, lengths

train_ds = LMDataset(train_texts, vocab)
val_ds   = LMDataset(val_texts, vocab)
test_ds  = LMDataset(test_texts, vocab)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=lm_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=lm_collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=lm_collate)

xb, yb, mb, lb = next(iter(train_loader))
print(xb.shape, yb.shape, mb.shape)


## 4. Implémentation successive : RNN simple → LSTM → GRU

On implémente un **unique modèle de langage paramétrable** (`RNNLanguageModel`) afin de comparer les trois
cellules récurrentes (`nn.RNN`, `nn.LSTM`, `nn.GRU`) dans des conditions strictement identiques
(même embedding, même taille cachée, même nombre de couches, même optimiseur).

Architecture : `Embedding → couche récurrente (RNN/LSTM/GRU) → Dropout → Linéaire (projection vers le vocabulaire)`.


In [ ]:
# ---------------------------------------------------------------------------
# 4.1 Modele de langage parametrable (RNN / LSTM / GRU)
# ---------------------------------------------------------------------------
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=1,
                 cell_type="lstm", pad_idx=0, dropout=0.2):
        super().__init__()
        self.cell_type = cell_type
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell_type]
        self.rnn = rnn_cls(emb_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)                 # (B, T, emb_dim)
        out, hidden = self.rnn(emb, hidden)      # out: (B, T, hidden_dim)
        out = self.drop(out)
        logits = self.fc(out)                    # (B, T, vocab_size)
        return logits, hidden

def init_weights_xavier(model):
    for name, p in model.named_parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
        else:
            nn.init.zeros_(p)

PAD_IDX = vocab.stoi[PAD]
VOCAB_SIZE = len(vocab)
print("Vocab size :", VOCAB_SIZE, "| pad idx :", PAD_IDX)


## 5. Rétropropagation à travers le temps (BPTT) et *gradient clipping*

**BPTT (Backpropagation Through Time) :** un réseau récurrent "déroulé" sur $T$ pas de temps équivaut à
un réseau profond de $T$ couches partageant les mêmes poids. Le gradient de la perte par rapport aux
paramètres se calcule en remontant la chaîne de dérivation à travers **tous** les pas de temps :

$$\frac{\partial \mathcal{L}}{\partial W} = \sum_{t=1}^{T} \frac{\partial \mathcal{L}_t}{\partial h_t}
\left(\prod_{k=t}^{2}\frac{\partial h_k}{\partial h_{k-1}}\right)\frac{\partial h_1}{\partial W}$$

Le produit de Jacobiennes $\prod \partial h_k/\partial h_{k-1}$ peut **exploser** ou **s'annuler**
(*vanishing/exploding gradients*) lorsque $T$ est grand, surtout pour un RNN simple — c'est précisément
la motivation des cellules à portes (LSTM/GRU).

**Gradient clipping :** pour éviter l'explosion du gradient, on borne la norme globale du gradient avant
la mise à jour des poids :

$$g \leftarrow g \cdot \min\left(1, \frac{\tau}{\|g\|}\right)$$

implémenté avec `torch.nn.utils.clip_grad_norm_`. Ci-dessous, on illustre **expérimentalement** son effet en
comparant la norme du gradient au fil de l'entraînement, avec et sans clipping, sur le RNN simple
(le plus sensible à ce phénomène).


In [ ]:
# ---------------------------------------------------------------------------
# 5.1 Boucle d'entrainement generique (1 epoque), avec/sans clipping,
#     enregistrement de la norme du gradient (illustration BPTT/clipping)
# ---------------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, criterion, clip=None, track_grad_norm=False):
    model.train()
    total_loss, total_tokens = 0.0, 0
    grad_norms = []
    for x, y, mask, lengths in loader:
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()

        if track_grad_norm:
            total_norm = torch.sqrt(sum((p.grad.detach() ** 2).sum() for p in model.parameters() if p.grad is not None))
            grad_norms.append(total_norm.item())

        if clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        n_tok = mask.sum().item()
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
    return total_loss / total_tokens, grad_norms

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for x, y, mask, lengths in loader:
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity


In [ ]:
# ---------------------------------------------------------------------------
# 5.2 Illustration experimentale : gradient clipping sur le RNN simple
# ---------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def make_model(cell_type):
    m = RNNLanguageModel(VOCAB_SIZE, emb_dim=128, hidden_dim=256, num_layers=1,
                          cell_type=cell_type, pad_idx=PAD_IDX).to(device)
    init_weights_xavier(m)
    return m

# Modele RNN simple, sans clipping
rnn_noclip = make_model("rnn")
opt = torch.optim.Adam(rnn_noclip.parameters(), lr=1e-3)
_, grad_norms_noclip = train_one_epoch(rnn_noclip, train_loader, opt, criterion, clip=None, track_grad_norm=True)

# Modele RNN simple, avec clipping (norme max = 5)
rnn_clip = make_model("rnn")
opt2 = torch.optim.Adam(rnn_clip.parameters(), lr=1e-3)
_, grad_norms_clip = train_one_epoch(rnn_clip, train_loader, opt2, criterion, clip=5.0, track_grad_norm=True)

plt.figure(figsize=(9,4))
plt.plot(grad_norms_noclip, label="Sans clipping", alpha=0.8)
plt.plot(grad_norms_clip, label="Avec clipping (max_norm=5)", alpha=0.8)
plt.xlabel("Itération"); plt.ylabel("Norme globale du gradient")
plt.title("Effet du gradient clipping sur la norme du gradient (RNN simple)")
plt.legend(); plt.yscale("log"); plt.grid(alpha=0.3)
plt.show()


**Lecture attendue du graphique :** sans clipping, la norme du gradient présente des pics importants
(parfois plusieurs ordres de grandeur), signe d'instabilité numérique typique des RNN simples sur des
séquences longues. Avec clipping, ces pics sont plafonnés, ce qui stabilise visiblement l'entraînement
et évite les divergences (loss = NaN).

## 6. Comparaison RNN / LSTM / GRU

On entraîne maintenant les trois architectures **dans des conditions identiques** (mêmes hyperparamètres,
mêmes données, gradient clipping activé pour tous) afin de comparer :
- la **stabilité** de l'entraînement (courbes de perte),
- la **performance** (perplexité de validation),
- la **capacité à mémoriser le contexte** (perplexité sur les séquences les plus longues du jeu de test),
- le **coût de calcul** (temps d'entraînement, nombre de paramètres).


In [ ]:
# ---------------------------------------------------------------------------
# 6.1 Entrainement comparatif RNN / LSTM / GRU (plusieurs epoques)
# ---------------------------------------------------------------------------
N_EPOCHS = 5
results = {}

for cell_type in ["rnn", "lstm", "gru"]:
    print(f"\n=== Entrainement : {cell_type.upper()} ===")
    model = make_model(cell_type)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_losses, val_ppls = [], []
    t0 = time.time()
    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, _ = train_one_epoch(model, train_loader, optimizer, criterion, clip=5.0)
        val_loss, val_ppl = evaluate(model, val_loader, criterion)
        train_losses.append(tr_loss); val_ppls.append(val_ppl)
        print(f"  epoch {epoch}: train_loss={tr_loss:.3f} | val_ppl={val_ppl:.2f}")
    train_time = time.time() - t0

    test_loss, test_ppl = evaluate(model, test_loader, criterion)
    results[cell_type] = {
        "model": model, "n_params": n_params, "train_time": train_time,
        "train_losses": train_losses, "val_ppls": val_ppls, "test_ppl": test_ppl,
    }
    print(f"  -> test perplexity = {test_ppl:.2f} | params = {n_params:,} | temps = {train_time:.1f}s")


In [ ]:
# ---------------------------------------------------------------------------
# 6.2 Tableau comparatif et courbes
# ---------------------------------------------------------------------------
summary = pd.DataFrame({
    "Modele": [k.upper() for k in results],
    "Parametres": [results[k]["n_params"] for k in results],
    "Temps entrainement (s)": [round(results[k]["train_time"], 1) for k in results],
    "Perplexite finale (val)": [round(results[k]["val_ppls"][-1], 2) for k in results],
    "Perplexite test": [round(results[k]["test_ppl"], 2) for k in results],
})
display(summary)

plt.figure(figsize=(9,4))
for k in results:
    plt.plot(results[k]["train_losses"], marker="o", label=k.upper())
plt.xlabel("Epoque"); plt.ylabel("Perte d'entrainement (cross-entropy)")
plt.title("Stabilite de l'entrainement : RNN vs LSTM vs GRU")
plt.legend(); plt.grid(alpha=0.3); plt.show()

plt.figure(figsize=(9,4))
for k in results:
    plt.plot(results[k]["val_ppls"], marker="o", label=k.upper())
plt.xlabel("Epoque"); plt.ylabel("Perplexite (validation)")
plt.title("Performance : RNN vs LSTM vs GRU")
plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 6.3 Capacite a memoriser le contexte : perplexite par tranche de longueur
# ---------------------------------------------------------------------------
@torch.no_grad()
def perplexity_by_length(model, dataset, bucket=(0,20,40,60)):
    model.eval()
    buckets = {f"{bucket[i]}-{bucket[i+1]}": [0.0, 0] for i in range(len(bucket)-1)}
    for x, y in [(d[0], d[1]) for d in dataset]:
        L = len(x)
        for i in range(len(bucket)-1):
            if bucket[i] <= L < bucket[i+1]:
                key = f"{bucket[i]}-{bucket[i+1]}"
                xb = x.unsqueeze(0).to(device); yb = y.unsqueeze(0).to(device)
                logits, _ = model(xb)
                loss = criterion(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
                buckets[key][0] += loss.item(); buckets[key][1] += 1
                break
    return {k: math.exp(v[0]/v[1]) if v[1] > 0 else None for k, v in buckets.items()}

for cell_type in results:
    ppl_by_len = perplexity_by_length(results[cell_type]["model"], test_ds)
    print(cell_type.upper(), "-> perplexite par longueur de sequence :", ppl_by_len)


**Interprétation attendue :** le RNN simple devrait montrer une **dégradation plus marquée** de la
perplexité sur les séquences longues (mémoire à court terme limitée par le *vanishing gradient*), tandis
que LSTM et GRU, grâce à leurs portes (*forget/input/output gate* pour LSTM, *update/reset gate* pour GRU),
maintiennent un contexte plus long plus efficacement. Le GRU, plus léger (moins de paramètres, pas de cellule
mémoire séparée), offre généralement un bon compromis performance/coût de calcul par rapport au LSTM.

## 7. Préparation des données — Partie B : Seq2Seq (Tatoeba eng-fra)

On utilise le corpus parallèle réel **eng-fra** (sous-ensemble Tatoeba, phrases courtes anglais → français),
téléchargé directement depuis l'hébergement officiel des tutoriels PyTorch. Étapes : nettoyage, filtrage
par longueur, tokenisation par mots, construction de **deux vocabulaires distincts** (source/anglais et
cible/français) avec tokens spéciaux `<pad> <unk> <bos> <eos>`, padding, masquage, et mini-lots.


In [ ]:
# ---------------------------------------------------------------------------
# 7.1 Telechargement et nettoyage du corpus parallele eng-fra
# ---------------------------------------------------------------------------
DATA_URL = "https://download.pytorch.org/tutorial/data.zip"
ZIP_PATH = "/content/data.zip"
EXTRACT_DIR = "/content/data_seq2seq"

if not os.path.exists(EXTRACT_DIR):
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

FRA_PATH = os.path.join(EXTRACT_DIR, "data", "eng-fra.txt")

def normalize(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Zàâäéèêëïîôöùûüç.!?']+", " ", s)
    return s.strip()

pairs = []
with open(FRA_PATH, encoding="utf-8") as f:
    for line in f:
        if "\t" not in line:
            continue
        eng, fra = line.strip().split("\t")[:2]
        eng, fra = normalize(eng), normalize(fra)
        if 1 <= len(eng.split()) <= 12 and 1 <= len(fra.split()) <= 12:
            pairs.append((eng, fra))

random.shuffle(pairs)
pairs = pairs[:20000]  # sous-echantillon pour un entrainement raisonnable sur Colab
print("Nombre de paires (apres filtrage):", len(pairs))
print(pairs[0])


In [ ]:
# ---------------------------------------------------------------------------
# 7.2 Vocabulaires source/cible avec tokens speciaux
# ---------------------------------------------------------------------------
def build_vocab(sentences, min_freq=2):
    counter = Counter(w for s in sentences for w in s.split())
    itos = list(SPECIALS) + [w for w, c in counter.items() if c >= min_freq]
    stoi = {w: i for i, w in enumerate(itos)}
    return itos, stoi

src_sentences = [p[0] for p in pairs]
tgt_sentences = [p[1] for p in pairs]

src_itos, src_stoi = build_vocab(src_sentences)
tgt_itos, tgt_stoi = build_vocab(tgt_sentences)
print("Vocab source (EN):", len(src_itos), "| Vocab cible (FR):", len(tgt_itos))

def encode_sentence(sentence, stoi):
    ids = [stoi.get(w, stoi[UNK]) for w in sentence.split()]
    return [stoi[BOS]] + ids + [stoi[EOS]]

n_total = len(pairs)
n_train = int(0.9 * n_total)
train_pairs = pairs[:n_train]
test_pairs  = pairs[n_train:]
print("Train pairs:", len(train_pairs), "| Test pairs:", len(test_pairs))


In [ ]:
# ---------------------------------------------------------------------------
# 7.3 Dataset / DataLoader Seq2Seq avec padding et masquage
# ---------------------------------------------------------------------------
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_stoi, tgt_stoi):
        self.pairs = pairs
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_ids = torch.tensor(encode_sentence(src, self.src_stoi), dtype=torch.long)
        tgt_ids = torch.tensor(encode_sentence(tgt, self.tgt_stoi), dtype=torch.long)
        return src_ids, tgt_ids

def translation_collate(batch):
    src_seqs, tgt_seqs = zip(*batch)
    src_pad = pad_sequence(src_seqs, batch_first=True, padding_value=src_stoi[PAD])
    tgt_pad = pad_sequence(tgt_seqs, batch_first=True, padding_value=tgt_stoi[PAD])
    src_mask = (src_pad != src_stoi[PAD])
    return src_pad, tgt_pad, src_mask

BATCH_SIZE_S2S = 64
train_s2s_ds = TranslationDataset(train_pairs, src_stoi, tgt_stoi)
test_s2s_ds  = TranslationDataset(test_pairs,  src_stoi, tgt_stoi)

train_s2s_loader = DataLoader(train_s2s_ds, batch_size=BATCH_SIZE_S2S, shuffle=True,  collate_fn=translation_collate)
test_s2s_loader  = DataLoader(test_s2s_ds,  batch_size=BATCH_SIZE_S2S, shuffle=False, collate_fn=translation_collate)

sb, tb, mb = next(iter(train_s2s_loader))
print(sb.shape, tb.shape, mb.shape)


## 8. Architecture encodeur–décodeur et *teacher forcing*

- **Encodeur :** un GRU bidirectionnel lit la phrase source et produit un état caché final résumant la phrase.
- **Décodeur :** un GRU unidirectionnel, initialisé avec le contexte de l'encodeur, génère la phrase cible
  token par token, en mode autoregressif.
- **Teacher forcing :** pendant l'entraînement, au lieu de réinjecter la prédiction du décodeur comme entrée
  du pas suivant, on réinjecte le **vrai token cible** (issu des données), avec une probabilité
  `teacher_forcing_ratio`. Cela stabilise et accélère l'entraînement, mais peut créer un **biais d'exposition**
  (*exposure bias*) : le modèle n'apprend pas à se corriger de ses propres erreurs, ce qui pénalise la
  génération au moment de l'inférence (où le vrai token n'est pas disponible).


In [ ]:
# ---------------------------------------------------------------------------
# 8.1 Encodeur (GRU bidirectionnel)
# ---------------------------------------------------------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)  # projection vers la taille du decodeur

    def forward(self, src):
        emb = self.embedding(src)
        outputs, hidden = self.gru(emb)               # hidden: (2, B, hidden_dim)
        hidden_cat = torch.cat([hidden[0], hidden[1]], dim=-1)  # (B, 2*hidden_dim)
        hidden_proj = torch.tanh(self.fc(hidden_cat)).unsqueeze(0)  # (1, B, hidden_dim)
        return outputs, hidden_proj

# ---------------------------------------------------------------------------
# 8.2 Decodeur (GRU unidirectionnel, pas a pas)
# ---------------------------------------------------------------------------
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_token, hidden):
        # input_token: (B, 1)
        emb = self.embedding(input_token)
        output, hidden = self.gru(emb, hidden)
        logits = self.fc(output.squeeze(1))   # (B, vocab_size)
        return logits, hidden

# ---------------------------------------------------------------------------
# 8.3 Modele Seq2Seq complet, avec teacher forcing
# ---------------------------------------------------------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_bos_idx, tgt_eos_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_bos_idx = tgt_bos_idx
        self.tgt_eos_idx = tgt_eos_idx
        self.device = device

    def forward(self, src, tgt=None, max_len=20, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        _, hidden = self.encoder(src)

        if tgt is not None:
            max_len = tgt.size(1)
            vocab_size = self.decoder.fc.out_features
            outputs = torch.zeros(batch_size, max_len, vocab_size, device=self.device)
            input_tok = tgt[:, 0].unsqueeze(1)  # <bos>
            for t in range(1, max_len):
                logits, hidden = self.decoder(input_tok, hidden)
                outputs[:, t, :] = logits
                use_tf = random.random() < teacher_forcing_ratio
                input_tok = tgt[:, t].unsqueeze(1) if use_tf else logits.argmax(1, keepdim=True)
            return outputs
        else:
            # inference (greedy) -- voir aussi greedy_decode / beam_search_decode plus bas
            input_tok = torch.full((batch_size, 1), self.tgt_bos_idx, dtype=torch.long, device=self.device)
            generated = []
            for _ in range(max_len):
                logits, hidden = self.decoder(input_tok, hidden)
                next_tok = logits.argmax(1, keepdim=True)
                generated.append(next_tok)
                input_tok = next_tok
            return torch.cat(generated, dim=1)


In [ ]:
# ---------------------------------------------------------------------------
# 8.4 Entrainement du modele Seq2Seq (BPTT + clipping + teacher forcing)
# ---------------------------------------------------------------------------
EMB_DIM, HID_DIM = 128, 256
encoder = Encoder(len(src_itos), EMB_DIM, HID_DIM, pad_idx=src_stoi[PAD]).to(device)
decoder = Decoder(len(tgt_itos), EMB_DIM, HID_DIM, pad_idx=tgt_stoi[PAD]).to(device)
seq2seq = Seq2Seq(encoder, decoder, tgt_stoi[BOS], tgt_stoi[EOS], device).to(device)

s2s_criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi[PAD])
s2s_optimizer = torch.optim.Adam(seq2seq.parameters(), lr=1e-3)

def train_s2s_epoch(model, loader, optimizer, criterion, clip=1.0, tf_ratio=0.5):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for src, tgt, src_mask in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        outputs = model(src, tgt, teacher_forcing_ratio=tf_ratio)
        loss = criterion(outputs[:, 1:].reshape(-1, outputs.size(-1)), tgt[:, 1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)   # BPTT + gradient clipping
        optimizer.step()
        n_tok = (tgt[:, 1:] != tgt_stoi[PAD]).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
    return total_loss / total_tokens

N_EPOCHS_S2S = 8
s2s_losses = []
for epoch in range(1, N_EPOCHS_S2S + 1):
    # on diminue progressivement le teacher forcing ratio (curriculum)
    tf_ratio = max(0.3, 0.8 - 0.05 * epoch)
    loss = train_s2s_epoch(seq2seq, train_s2s_loader, s2s_optimizer, s2s_criterion,
                            clip=1.0, tf_ratio=tf_ratio)
    s2s_losses.append(loss)
    print(f"epoch {epoch}: loss={loss:.3f} | perplexite={math.exp(loss):.2f} | tf_ratio={tf_ratio:.2f}")

plt.figure(figsize=(7,4))
plt.plot(s2s_losses, marker="o")
plt.xlabel("Epoque"); plt.ylabel("Perte (cross-entropy)")
plt.title("Entrainement du modele Seq2Seq (encodeur-decodeur)")
plt.grid(alpha=0.3); plt.show()


## 9. Stratégies de décodage : glouton (*greedy*) et *beam search*

- **Décodage glouton :** à chaque pas, on choisit le token le plus probable (`argmax`). Simple et rapide,
  mais **myope** : une erreur locale tôt dans la séquence peut compromettre toute la suite
  (pas de retour en arrière possible).
- **Beam search :** on conserve à chaque étape les $k$ (largeur de faisceau) séquences partielles les plus
  probables (score = somme des log-probabilités, avec normalisation par la longueur pour ne pas favoriser
  les séquences courtes), et on élargit chacune avant de retenir les $k$ meilleures. Cela explore un espace
  de recherche plus large et donne en général une meilleure qualité de traduction, au prix d'un coût de
  calcul plus élevé (proportionnel à $k$).


In [ ]:
# ---------------------------------------------------------------------------
# 9.1 Decodage glouton
# ---------------------------------------------------------------------------
@torch.no_grad()
def greedy_decode(model, src_sentence, max_len=20):
    model.eval()
    src_ids = torch.tensor([encode_sentence(src_sentence, src_stoi)], dtype=torch.long, device=device)
    _, hidden = model.encoder(src_ids)
    input_tok = torch.tensor([[tgt_stoi[BOS]]], dtype=torch.long, device=device)
    output_ids = []
    for _ in range(max_len):
        logits, hidden = model.decoder(input_tok, hidden)
        next_id = logits.argmax(1).item()
        if next_id == tgt_stoi[EOS]:
            break
        output_ids.append(next_id)
        input_tok = torch.tensor([[next_id]], dtype=torch.long, device=device)
    return [tgt_itos[i] for i in output_ids]

# ---------------------------------------------------------------------------
# 9.2 Decodage par faisceau (beam search)
# ---------------------------------------------------------------------------
@torch.no_grad()
def beam_search_decode(model, src_sentence, beam_width=3, max_len=20, length_penalty=0.7):
    model.eval()
    src_ids = torch.tensor([encode_sentence(src_sentence, src_stoi)], dtype=torch.long, device=device)
    _, hidden0 = model.encoder(src_ids)

    # chaque hypothese : (sequence_ids, score_log_prob_cumule, hidden, terminee)
    beams = [([tgt_stoi[BOS]], 0.0, hidden0, False)]

    for _ in range(max_len):
        all_candidates = []
        for seq, score, hidden, done in beams:
            if done:
                all_candidates.append((seq, score, hidden, True))
                continue
            input_tok = torch.tensor([[seq[-1]]], dtype=torch.long, device=device)
            logits, new_hidden = model.decoder(input_tok, hidden)
            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
            top_log_probs, top_ids = log_probs.topk(beam_width)
            for lp, idx in zip(top_log_probs.tolist(), top_ids.tolist()):
                new_seq = seq + [idx]
                new_done = (idx == tgt_stoi[EOS])
                all_candidates.append((new_seq, score + lp, new_hidden, new_done))

        # normalisation par la longueur (length penalty) pour ne pas favoriser les sequences courtes
        def norm_score(c):
            seq, score, _, _ = c
            return score / (len(seq) ** length_penalty)

        all_candidates.sort(key=norm_score, reverse=True)
        beams = all_candidates[:beam_width]
        if all(done for _, _, _, done in beams):
            break

    best_seq = max(beams, key=norm_score)[0]
    ids = [i for i in best_seq[1:] if i not in (tgt_stoi[EOS],)]
    return [tgt_itos[i] for i in ids]

# Exemples qualitatifs
for example_src, example_tgt in test_pairs[:5]:
    greedy_out = " ".join(greedy_decode(seq2seq, example_src))
    beam_out   = " ".join(beam_search_decode(seq2seq, example_src, beam_width=3))
    print(f"SRC      : {example_src}")
    print(f"REF      : {example_tgt}")
    print(f"GREEDY   : {greedy_out}")
    print(f"BEAM(k=3): {beam_out}")
    print("-" * 60)


## 10. Évaluation : perplexité et BLEU

- La **perplexité** (déjà calculée pour les modèles de langage de la Partie A) évalue la qualité d'un modèle
  probabiliste de séquence.
- Pour la **traduction**, on utilise le **score BLEU** (*Bilingual Evaluation Understudy*), qui combine une
  précision n-gramme (1 à 4-grammes) entre la traduction produite et la référence, pondérée par une pénalité
  de brièveté (*brevity penalty*) pour éviter de favoriser des traductions trop courtes.

On compare ci-dessous le score BLEU obtenu par le **décodage glouton** et par le **beam search**.


In [ ]:
# ---------------------------------------------------------------------------
# 10.1 BLEU : glouton vs beam search sur un sous-ensemble de test
# ---------------------------------------------------------------------------
N_EVAL = 200
eval_pairs = test_pairs[:N_EVAL]

references, hyps_greedy, hyps_beam = [], [], []
for src_sent, tgt_sent in eval_pairs:
    references.append([tgt_sent.split()])
    hyps_greedy.append(greedy_decode(seq2seq, src_sent))
    hyps_beam.append(beam_search_decode(seq2seq, src_sent, beam_width=3))

smoothing = SmoothingFunction().method4
bleu_greedy = corpus_bleu(references, hyps_greedy, smoothing_function=smoothing)
bleu_beam   = corpus_bleu(references, hyps_beam,   smoothing_function=smoothing)

print(f"BLEU (decodage glouton)  : {bleu_greedy*100:.2f}")
print(f"BLEU (beam search, k=3)  : {bleu_beam*100:.2f}")

bleu_summary = pd.DataFrame({
    "Strategie de decodage": ["Glouton", "Beam search (k=3)"],
    "BLEU (%)": [round(bleu_greedy*100, 2), round(bleu_beam*100, 2)],
})
display(bleu_summary)


## 11. Analyse critique

**RNN vs LSTM vs GRU :**
- Le **RNN simple** est le plus rapide à entraîner (moins de paramètres, pas de mécanisme de porte) mais
  le moins stable : la norme du gradient explose plus facilement (cf. section 5), et sa perplexité se dégrade
  davantage sur les séquences longues (mémoire à court terme limitée).
- Le **LSTM**, grâce à sa cellule mémoire et ses trois portes (*input, forget, output*), gère mieux les
  dépendances longues, au prix d'un nombre de paramètres et d'un temps de calcul plus élevés.
- Le **GRU**, avec seulement deux portes (*update, reset*) et sans cellule mémoire séparée, offre un
  compromis souvent très efficace : performance proche du LSTM pour un coût de calcul réduit.

**Gradient clipping :** indispensable pour stabiliser l'entraînement de réseaux récurrents profonds /
sur de longues séquences ; sans lui, l'entraînement du RNN simple peut diverger (perte explosant vers
l'infini ou NaN).

**Teacher forcing :** accélère et stabilise l'entraînement du Seq2Seq, mais induit un biais d'exposition :
le modèle voit toujours le bon contexte pendant l'entraînement, alors qu'en inférence il doit se baser sur
ses propres prédictions, potentiellement erronées. Réduire progressivement le ratio de teacher forcing
(*curriculum learning*) atténue partiellement ce problème.

**Glouton vs beam search :** le beam search améliore généralement le score BLEU par rapport au décodage
glouton, car il explore plusieurs hypothèses au lieu de s'engager définitivement sur le choix localement
optimal à chaque pas. Le gain reste toutefois modéré sur de petites phrases (où peu d'ambiguïté existe),
et le coût de calcul croît linéairement avec la largeur de faisceau $k$.

**Limites du dispositif expérimental :**
- corpus de taille réduite (sous-échantillonnage pour un temps de calcul raisonnable sur Colab) ;
- absence de mécanisme d'**attention** dans le décodeur (le contexte est résumé en un seul vecteur, ce qui
  limite la qualité sur les phrases longues — un problème bien documenté, motivant les architectures
  encodeur-décodeur avec attention, puis les Transformers) ;
- tokenisation au mot (pas de sous-mots / BPE), ce qui multiplie les tokens `<unk>` sur le vocabulaire rare.

**Pistes d'amélioration :** ajouter une couche d'attention (Bahdanau/Luong) au décodeur, utiliser une
tokenisation par sous-mots (BPE/SentencePiece), augmenter la taille du corpus et le nombre d'époques,
et comparer avec une architecture Transformer.


## 12. Question de synthèse — Partie III

> *Dans quelle mesure les architectures récurrentes permettent-elles de modéliser efficacement une séquence
> réelle, et comment justifier le passage d'un RNN simple vers un LSTM/GRU puis vers un schéma
> encodeur–décodeur pour une tâche de génération ou de traduction ?*

Les architectures récurrentes modélisent une séquence en factorisant sa probabilité jointe par la règle de
chaîne et en résumant le passé dans un état caché mis à jour à chaque pas de temps. Cette formulation est
naturellement adaptée aux données réelles ordonnées dans le temps (texte, parole, séries temporelles),
puisqu'elle respecte à la fois la **causalité** (chaque prédiction ne dépend que du passé) et la
**dépendance séquentielle** entre éléments voisins.

Le **RNN simple** constitue la formulation la plus directe de cette idée, mais sa capacité de mémorisation
est en pratique limitée par le phénomène de **vanishing/exploding gradient** lors de la rétropropagation à
travers le temps (BPTT) : sur nos expériences (section 5 et 6), il se montre numériquement instable (pics de
norme de gradient) et perd en précision sur les séquences longues. Le passage au **LSTM** introduit une
cellule mémoire interne protégée par des portes multiplicatives (*forget, input, output*), qui permettent au
gradient de circuler sur de longues distances temporelles sans s'atténuer exponentiellement — d'où une
perplexité plus stable sur les séquences longues dans nos résultats. Le **GRU** simplifie ce mécanisme (deux
portes au lieu de trois, pas de cellule séparée) tout en conservant l'essentiel du bénéfice, pour un coût de
calcul réduit : c'est un compromis pertinent lorsque les ressources sont limitées, comme c'est souvent le cas
en contexte pédagogique ou en déploiement léger.

Cependant, un simple modèle récurrent unidirectionnel reste insuffisant pour des tâches de **génération
conditionnelle** comme la traduction automatique, où la sortie dépend de l'**intégralité** d'une séquence
source de longueur différente de la séquence cible. Le schéma **encodeur–décodeur** répond précisément à ce
besoin : l'encodeur lit toute la phrase source et produit une représentation condensée (le contexte), que le
décodeur utilise pour générer la séquence cible token par token, de façon autoregressive. L'usage du
**teacher forcing** pendant l'entraînement accélère la convergence en ancrant le décodeur sur les vraies
valeurs cibles, au prix d'un biais d'exposition qu'il faut atténuer (par exemple en réduisant progressivement
le ratio de teacher forcing).

Enfin, la **qualité du décodage** à l'inférence dépend fortement de la stratégie choisie : le décodage
glouton est rapide mais myope, tandis que le **beam search** explore plusieurs hypothèses simultanément et
améliore en général le score BLEU mesuré expérimentalement, en contrepartie d'un coût de calcul plus élevé.

**Limites observées expérimentalement :** la principale limite du schéma encodeur–décodeur sans attention
est la compression de toute la phrase source en un vecteur de taille fixe, ce qui dégrade la qualité sur les
phrases longues — une limite que seul un mécanisme d'attention (puis les architectures Transformer) permet
de dépasser efficacement. Ainsi, le passage progressif RNN → LSTM/GRU → encodeur–décodeur illustre une
adaptation croissante de l'architecture à la complexité réelle de la tâche : d'abord résoudre le problème de
mémoire à long terme, puis celui de la correspondance entre deux séquences de longueurs différentes.


## Conclusion de la Partie III

Ce notebook a couvert, sur des données textuelles réelles (IMDb et Tatoeba eng-fra) :
1. la formulation probabiliste d'un modèle de langage et la notion de perplexité ;
2. l'implémentation comparée de RNN, LSTM et GRU ;
3. le principe du BPTT et l'illustration expérimentale du gradient clipping ;
4. la préparation rigoureuse des données séquentielles (tokenisation, vocabulaire, tokens spéciaux, padding,
   masquage, mini-lots) ;
5. un système Seq2Seq encodeur–décodeur avec teacher forcing ;
6. deux stratégies de décodage (glouton, beam search) ;
7. une évaluation quantitative (perplexité, BLEU) et une analyse critique des résultats.

**Pour le rapport final**, il reste à : exporter les courbes/tableaux générés ici dans l'annexe expérimentale,
rédiger la section "Partie III" du rapport scientifique en reprenant l'analyse critique et la question de
synthèse ci-dessus, et les relier à la discussion transversale finale (comment le deep learning adapte ses
architectures à la structure des données : tabulaire → MLP, image → CNN, séquentielle → RNN/LSTM/GRU/Seq2Seq).
